# Ejemplo completo y guiado de fine-tuning

Se va a hacer fine-tuning para ajustar un modelo de BERT a una tarea de análisis de sentimientos sobre opiniones de películas con un dataset
de Huggingface.

Para ello, previamente hay que instalar tanto la librería transformers de Huggingface.


In [1]:
!pip install --upgrade fsspec

In [2]:
!pip freeze

absl-py==2.3.0
aiohappyeyeballs @ file:///home/conda/feedstock_root/build_artifacts/aiohappyeyeballs_1741775197943/work
aiohttp @ file:///home/conda/feedstock_root/build_artifacts/aiohttp_1749923405795/work
aiosignal @ file:///home/conda/feedstock_root/build_artifacts/aiosignal_1734342155601/work
annotated-types @ file:///home/conda/feedstock_root/build_artifacts/annotated-types_1733247046149/work
asttokens @ file:///home/conda/feedstock_root/build_artifacts/asttokens_1733250440834/work
astunparse==1.6.3
async-timeout @ file:///home/conda/feedstock_root/build_artifacts/async-timeout_1733235340728/work
attrs @ file:///home/conda/feedstock_root/build_artifacts/attrs_1741918516150/work
blis==1.3.0
Brotli @ file:///home/conda/feedstock_root/build_artifacts/brotli-split_1749229842835/work
cachetools==6.1.0
catalogue @ file:///home/conda/feedstock_root/build_artifacts/catalogue_1736092400301/work
certifi @ file:///home/conda/feedstock_root/build_artifacts/certifi_1749972191589/work/certifi
c

In [3]:
!pip install -q transformers[torch] datasets

In [8]:
from datasets import load_dataset_builder
import tqdm 
ds = load_dataset_builder("rotten_tomatoes")

print("Descripción del dataset: ", ds.info.description)
print("Características del dataset: ", ds.info.features)

ValueError: Invalid pattern: '**' can only be an entire path component

## 1) Consultar datos del dataset

La función load_dataset_builder permite obtener información asociada al dataset sin descargarlo fisicamente al entorno de trabajo.

In [ ]:
# Hay que ver si el dataset ya tiene las particiones hechas o las tenemos que hacer nosotros
from datasets import get_dataset_split_names
get_dataset_split_names("rotten_tomatoes")

['train', 'validation', 'test']

## 2) Cargar el dataset

In [ ]:
from datasets import load_dataset

# se carga el dataset completo
dataset = load_dataset("rotten_tomatoes")

labels = dataset['train'].features['label'].names
NUM_LABELS = len(labels)
print('Labels: ', labels, ', número de labels: ', NUM_LABELS)

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Labels:  ['neg', 'pos'] , número de labels:  2


## 3) Tokenización

Hay que preparar los textos para que puedan tener el formato de entrada que requiere el transformer que usemos. Vamos a utilizar el modelo BERT, en este
caso la versión uncased del modelo bert_base.

In [ ]:
from transformers import AutoTokenizer

model_id = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# Como las redes neuronales necesitan que todas las entradas tengan el mismo tamaño (mismo número de tokens). El modelo BERT admite un tamaño máximo
# de 512 tokens

# Es recomendable ver cuál es el tamaño máximo en el dataset que utilizamos. Vemos cuál es el texto con mayor número de tokens.
MAX_LENGTH = max([len(tokenizer(text).input_ids) for text in dataset['train']['text']])
print("Tamaño máximo en el train: ",MAX_LENGTH)

Tamaño máximo en el train:  78


Si la el tamaño máximo en el train es mucho menor que el tamaño máximo del modelo se tiene que ajustar a ese tamaño máximo para no desperdiciar memoria. Otra opción es estudiar la distribución de los tamaños de los textos y elegir un tamaño asociado a un percentil alto (80 o 90 por ejemplo) para asegurar que es un tamaño que tiene la mayoría de los textos Se puede analizar también la distribución de la colección.

La tokenización por lotes (batch) es más eficiente computacionalmente que si tokenizamos la colección entera de una vez.

In [ ]:
# La función recibe un lote de instancias y se tokeniza el campo texto. El padding es necesario tomando como referencia el tamaño máximo
# A todas las sentencias más cortas del tamaño máximo se les irá añadiendo tokens pad al final, tokens 0. No se hace truncation porque hemos definido como
# longitud máxima la del tamaño máximo de una sentencia en el dataset => no tiene sentido truncar aquí
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length",max_length=MAX_LENGTH)

In [ ]:
# se tokeniza todo el dataset por lotes
# sobre el diccionario con los tres split se llama a map para que aplique la función tokenize a cada split y al hacer batched a True se le está diciendo
# que lo haga por lotes
encoded_data = dataset.map(tokenize, batched=True)
encoded_data

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
})

In [ ]:
# Para aliviar el entrenamiento, nos quedamos solo con una parte del dataset
small_train_dataset = encoded_data["train"].shuffle(seed=42).select(range(1000))
small_validation_dataset = encoded_data["validation"].shuffle(seed=42).select(range(500))
small_test_dataset = encoded_data["test"].shuffle(seed=42).select(range(500))

full_train_dataset = encoded_data["train"]
full_validation_dataset = encoded_data["validation"]
full_test_dataset = encoded_data["test"]

In [ ]:
# se puede comprobar que todos los textos tienen el mismo tamaño
import random
for i in range(10):
    index = random.randint(0,small_train_dataset.num_rows)
    print('text:', index, ' len:', len(small_train_dataset[index]['input_ids']))


text: 746  len: 78
text: 417  len: 78
text: 515  len: 78
text: 840  len: 78
text: 268  len: 78
text: 745  len: 78
text: 973  len: 78
text: 847  len: 78
text: 787  len: 78
text: 392  len: 78


## 4) Fine-tuning modelo preentrenado

Lo primero es cargar el modelo que hay que ajustar, que será el modelo de BERT base en su versión uncased. La clase AutoModelForSequenceClassification permite cargar dicho modelo ya extendido para la tarea de clasificación de secuencias. Este modelo extendido tiene una última capa softmax que obtiene una probabilidad por cada clase, en este problema será una clasificación binaria. Además del modelo es necesario hay que indicar el número de clases (labels).

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=NUM_LABELS)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


La celda anterior avisa de que algunos parámetros no han sido ajustados, pero esto es algo normal.

### Hiperparámetros

Hay que definir los hiperparámetros e emplear durante el entrenamiento (número de epochs para el entrenamiento, tamaño del lote (batch), ratio de aprendizaje (learning rate), etc.). Para encontrar la configuración óptima para cada tarea lo habitual es experimentar y probar. En este ejemplo se va a trabajar con los hipérparametros por defecto. ¿Cómo saber cuáles son? Si se instancia un objeto de la clase TrainingArguments se pueden ver todos los hiperparámetros ajustables con los valores con los que están inicializados por defecto.

In [ ]:
from transformers import TrainingArguments
args = TrainingArguments(output_dir="./outputs")
args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,


In [ ]:
# Vamos a modificar el lote para el tamaño del batch para entrenamiento y la validación
args.per_device_train_batch_size = 32
args.per_device_eval_batch_size = 32


# Y también modificamos la estrategia de entrenamiento del modelo. La estrategia puede ser epoch o step. Los pasos (steps) se refieren a la cantidad
# de lotes procesados por el modelo durante el entrenamiento, mientras que las épocas se refieren a un paso completo por todo el conjunto de datos
# de entrenamiento. Vamos a indicar que por épocas.
args.evaluation_strategy="epoch"

# lo que se le está diciendo con esta línea es que no envíe los logs a ningún sitio. Colab por defecto está añadiendo "wandb" en los hiperparámetros
# report_to=['tensorboard', 'wandb'] y eso luego da problemas en el entrenamiento. Al indicar "none" es como que se desactiva.
args = TrainingArguments(output_dir="./outputs", report_to="none")

## 5) Definir el conjunto de métricas

Es importante decidir el conjunto de métricas que se van a utilizar para evaluar el modelo sobre el conjunto de evaluación y ver si está aprendiendo
el modelo o no para ajustar los parámetros en función de los resultados.

Las métricas dependen de cada tarea. Para clasificación de textos es habitual utilizar accuracy, precision, recall y f-measure.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    y_true = pred.label_ids # labels reales
    y_pred = pred.predictions.argmax(-1) # predicciones

    accuracy = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(y_true,y_pred, average='macro')

    return {
        'accuracy: ', accuracy,
        'f1: ', f1,
        'precision: ', precision,
        'recall: ', recall
    }

## 6) Entrenamiento

Para entrenar el modelo hay que crear un objeto de la clase Trainer. Esta clase de Pytorch está optimizada para entrenar transformers y nos ahorra
trabajo porque no es necesario escribir el ciclo de entrenamiento por epochs y calcular las métricas sobre el conjunto de validación.

In [ ]:
# Para crear un objeto de tipo Trainer hay que pasar el modelo, los conjuntos de entrenamiento y validación, los argumentos y la función para calcular
# las métricas.

from transformers import Trainer

trainer = Trainer(
    model = model,            # modelo que será ajustado
    #train_dataset = encoded_data['train'], # conjunto training
    train_dataset = small_train_dataset,
    #eval_dataset = encoded_data['validation'],   # conjunto de validación
    eval_dataset = small_train_dataset,

    args = args,     # hiperparámetros
    compute_metrics=compute_metrics,    # función para computar las métricas
)

In [ ]:
# Se lanza el entrenamiento. Esto puede tardar unos minutos.
trainer.train()

Step,Training Loss


TrainOutput(global_step=375, training_loss=0.052127578735351565, metrics={'train_runtime': 65.944, 'train_samples_per_second': 45.493, 'train_steps_per_second': 5.687, 'total_flos': 120249974520000.0, 'train_loss': 0.052127578735351565, 'epoch': 3.0})

In [ ]:
# Una vez finalizado el entrenamiento, es posible evaluar el modelo final sobre el conjunto de datos de validación.
trainer.evaluate()

TypeError: 'set' object does not support item assignment

## 6) Evaluación

El modelo entrenado se utilizar para predecir las clases de los textos del conjunto de test. Sobre la salida del modelo se aplica la función softmax, que calcula la probabilidad de cada clase y para devolver la que tiene mayor probabilidad utilizamos la función argmax.